# 物流延误与客户评价/体验


基于 Olist 的物流延误对评价/体验的影响分析。

**口径说明**：延误率、差评率等订单级指标使用 `v_order_delay`（按 `order_id` 去重一单一行），避免宽表 JOIN 膨胀。


## 1. 环境准备


In [ ]:
import duckdb
import os
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False


from pathlib import Path
import os

当前目录 = Path.cwd().resolve()
项目目录 = 当前目录.parent if 当前目录.name == "notebooks" else 当前目录
数据库路径 = 项目目录 / "data" / "processed" / "olist.duckdb"
输出目录 = 项目目录 / "outputs" / "figures"
输出目录.mkdir(parents=True, exist_ok=True)
os.chdir(项目目录)

import runpy
runpy.run_path("scripts/data_cleaning.py", run_name="__main__")
con = duckdb.connect(str(数据库路径), read_only=True)
print("db:", 数据库路径)
print("订单级 v_order_facts:", con.execute("SELECT COUNT(*) FROM v_order_facts").fetchone()[0])
print("已送达订单级 v_order_delay:", con.execute("SELECT COUNT(*) FROM v_order_delay").fetchone()[0])


## 2. 延误概览


In [ ]:
# 订单级：按 order_id 去重后的 v_order_delay
核心指标 = con.execute('''
SELECT
    ROUND(100.0 * SUM(is_delayed) / COUNT(*), 2) AS 延误率百分比,
    ROUND(AVG(CASE WHEN is_delayed = 1 THEN delay_days END), 1) AS 延误订单平均延误天数,
    COUNT(*) AS 已送达订单数
FROM v_order_delay
''').fetchdf()


In [ ]:
print(核心指标.to_string(index=False))


## 3. 延误 vs 准点评分


In [ ]:
延误评价对比 = con.execute('''
SELECT
    is_delayed AS 是否延误,
    COUNT(*) AS 订单数,
    ROUND(AVG(review_score), 2) AS 平均评分,
    ROUND(100.0 * SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) / COUNT(*), 2) AS 差评率百分比
FROM v_order_delay
WHERE review_score IS NOT NULL
GROUP BY is_delayed
''').fetchdf()


In [ ]:
print(延误评价对比.to_string(index=False))


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

test_df = con.execute("""
    SELECT
        is_delayed,
        COUNT(*) AS 订单数,
        SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) AS 差评订单数
    FROM v_order_delay
    WHERE review_score IS NOT NULL
    GROUP BY is_delayed
    ORDER BY is_delayed
""").fetchdf()

counts = test_df["差评订单数"].to_numpy()
nobs = test_df["订单数"].to_numpy()

z_stat, p_value = proportions_ztest(
    count=counts,
    nobs=nobs,
    alternative="two-sided"
)

print(test_df)
print(f"\nZ 统计量：{z_stat:.2f}")
print(f"P 值：{p_value:.6g}")

In [ ]:
from math import sqrt
from scipy.stats import norm

未延误差评率 = counts[0] / nobs[0]
延误差评率 = counts[1] / nobs[1]

差评率差异 = 延误差评率 - 未延误差评率

标准误 = sqrt(
    延误差评率 * (1 - 延误差评率) / nobs[1]
    + 未延误差评率 * (1 - 未延误差评率) / nobs[0]
)

临界值 = norm.ppf(0.975)  # 95% 置信水平对应约 1.96

下限 = 差评率差异 - 临界值 * 标准误
上限 = 差评率差异 + 临界值 * 标准误

print(f"未延误差评率：{未延误差评率 * 100:.2f}%")
print(f"延误差评率：{延误差评率 * 100:.2f}%")
print(f"差评率差异：{差评率差异 * 100:.2f} 个百分点")
print(f"95% 置信区间：[ {下限 * 100:.2f}，{上限 * 100:.2f} ] 个百分点")

## 4. 容忍度曲线


In [ ]:
延误分组数据 = con.execute("""
    SELECT
        CASE
            WHEN is_delayed = 0 THEN '按时或提前'
            WHEN delay_days = 0 THEN '延误不足1天'
            WHEN delay_days <= 2 THEN '延误1–2天'
            WHEN delay_days <= 5 THEN '延误3–5天'
            WHEN delay_days <= 10 THEN '延误6–10天'
            ELSE '延误11天及以上'
        END AS 延误分组,

        MIN(delay_days) AS 排序用最小延误天数,

        ROUND(
            AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) * 100,
            2
        ) AS 差评率百分比

    FROM v_order_delay
    WHERE review_score IS NOT NULL
    GROUP BY 延误分组
    ORDER BY 排序用最小延误天数
""").fetchdf()

plt.figure(figsize=(10, 5))

plt.plot(
    延误分组数据["延误分组"].to_numpy(),
    延误分组数据["差评率百分比"].to_numpy(),
    marker="o",
    linewidth=2.5,
    color="#E76F51"
)

plt.title("延误天数与差评率：客户容忍度阈值", fontsize=14, fontweight="bold")
plt.xlabel("延误分组")
plt.ylabel("差评率（%）")
plt.ylim(0, 100)
plt.grid(axis="y", alpha=0.3)

for x, y in zip(延误分组数据["延误分组"], 延误分组数据["差评率百分比"]):
    plt.text(x, y + 3, f"{y}%", ha="center")

plt.tight_layout()
os.makedirs("outputs", exist_ok=True)
plt.savefig(输出目录 / "tolerance_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. 同品类分层对比


In [ ]:
# 订单×品类去重，避免支付/评价 JOIN 膨胀
品类评分对比 = con.execute('''
WITH 订单品类 AS (
    SELECT
        品类.品类,
        延误.is_delayed,
        延误.review_score
    FROM v_order_category 品类
    JOIN v_order_delay 延误
        ON 品类.order_id = 延误.order_id
    WHERE 延误.review_score IS NOT NULL
)
SELECT
    品类,
    COUNT(*) AS 订单数,
    ROUND(
        AVG(CASE WHEN is_delayed = 1 THEN review_score END)
        - AVG(CASE WHEN is_delayed = 0 THEN review_score END),
        2
    ) AS 评分差异
FROM 订单品类
GROUP BY 品类
HAVING COUNT(*) >= 200
ORDER BY 评分差异 ASC
''').fetchdf()


In [ ]:
print(f'共 {len(品类评分对比)} 个品类')


In [ ]:
print(品类评分对比.head(10).to_string(index=False))
os.makedirs('outputs', exist_ok=True)
fig, ax = plt.subplots(figsize=(10, 7))
评分差异值 = 品类评分对比.head(15)['评分差异'].values
品类标签 = 品类评分对比.head(15)['品类'].values
颜色 = ['crimson' if x < 0 else 'seagreen' for x in 评分差异值]
ax.barh(品类标签[::-1], 评分差异值[::-1], color=颜色[::-1], edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('同品类延误 vs 准点评分差异（订单×品类去重）')
plt.tight_layout()
plt.savefig(输出目录 / 'category_delay_impact.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
con.close()
